# CXR Insight — train the multi-label CNN on free Colab

Research prototype; not for clinical use. Trains EfficientNet-B0 on NIH ChestX-ray14
(14 findings, multi-label; "No Finding" = all labels 0). Data: NIH Clinical Center,
https://nihcc.app.box.com/v/ChestXray-NIHCC — cite Wang et al., CVPR 2017 and
acknowledge the NIH Clinical Center. Checkpoints go to Google Drive, so an interrupted
session resumes by re-running the training cell with the same arguments.

Runtime → Change runtime type → T4 GPU (free tier).

In [ ]:
# Clone the code and install it. Pin a commit for reported runs.
!git clone https://github.com/devbyabir/cxr-insight.git
%cd cxr-insight
!pip install -q -e ".[cnn]"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN = "/content/drive/MyDrive/cxr-insight/runs/cnn_nih_seed42"  # persistent across sessions
PARTS = 1   # number of NIH image archives (1-12). Fewer parts = subset results only.

In [ ]:
# Download official NIH metadata and the chosen image archives (Hugging Face mirror
# of the NIH Box files), then build patient-grouped splits.
import os, subprocess
B = "https://huggingface.co/datasets/alkzar90/NIH-Chest-X-ray-dataset/resolve/main/data"
os.makedirs("data/raw/nih/images", exist_ok=True)
for f in ["Data_Entry_2017_v2020.csv", "test_list.txt", "train_val_list.txt"]:
    subprocess.run(["wget", "-q", "-nc", "-O", f"data/raw/nih/{f}", f"{B}/{f}"], check=True)
for i in range(1, PARTS + 1):
    z = f"data/raw/nih/images_{i:03d}.zip"
    subprocess.run(["wget", "-q", "-nc", "-O", z, f"{B}/images/images_{i:03d}.zip"], check=True)
    subprocess.run(["unzip", "-q", "-n", z, "-x", "__MACOSX/*", "-d", "data/raw/nih/images"], check=True)
    os.remove(z)

In [ ]:
!cxr-nih-prepare --data-entry data/raw/nih/Data_Entry_2017_v2020.csv --test-list data/raw/nih/test_list.txt --image-dir data/raw/nih/images --output data/splits/nih_seed42

In [ ]:
# Re-run this cell after a disconnect: it resumes from RUN/last.pt.
!cxr-train-cnn --splits data/splits/nih_seed42 --image-dir data/raw/nih/images --out "$RUN" --epochs 8 --batch-size 64 --workers 2

In [ ]:
# Evaluate ONCE on the frozen test split, after all choices are fixed.
!cxr-evaluate-cnn --checkpoint "$RUN/classifier.pt" --splits data/splits/nih_seed42 --image-dir data/raw/nih/images --out "$RUN/test_eval"